In [ ]:
from datetime import datetime
from getpass import getpass
import random

rdm_url = 'staging.rdm.example.com'
idp_name_1 = None  # 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-WikiEditing-{}'.format(datetime.now().strftime('%Y%m%d'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_wiki_collaboration = False
new_wikiname = 'New Wiki'

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WikiEditing

- サブシステム名: アドオン
- ページ/アドオン: WikiEditing
- 機能分類: Wiki操作
- シナリオ名: WikiEditing
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## 対象のプロジェクトをクリックする

プロジェクトが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## Wikiタブをクリックする

画面がWikiに切り替わること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Wiki", exact=True).click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "プロジェクトのWiki"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左のプロジェクトのWikiの上にある「新規」ボタンをクリックする

新規Wikiページを追加画面が表示される

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#newWiki"]').click() 
    await expect(page.locator('h3.modal-title', has_text="新規Wikiページを追加")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 同階層に追加を選択し、任意の新規Wiki名を入力して「追加」ボタンをクリックする

画面左のプロジェクトのWikiに、追加したWikiが表示される

In [ ]:
async def _step(page):
    await page.locator('input[name="addHierarchy"][value="same"]').check()
    await page.fill('#data', new_wikiname)
    await page.locator('#add-wiki-submit').click()
    await expect(page.locator(f'//*[contains(@class, "title-text")]//*[text() = "{new_wikiname}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プロジェクトのWiki」内にある追加したWikiを選択

Wikiの内容が表示されること（初期表示は空白で「重要な情報、リンク、または画像をここに追加して、プロジェクトを説明します。」の記載があること）

In [ ]:
async def _step(page):
    await grdm.open_wiki(page, new_wikiname, '重要な情報、リンク、または画像をここに追加して、プロジェクトを説明します。')

await run_pw(_step)

## 「編集」ボタンをクリックする

編集欄に、試験用Wikiに保存済みの「編集前の本文」が表示されること。

In [ ]:
initial_wiki_text = '編集前の本文'

async def _step(page):
    wiki_url = await page.evaluate(
        'new URL(window.contextVars.wiki.urls.page, window.location.href).href'
    )
    response = await page.request.post(wiki_url, data={'markdown': initial_wiki_text + '\n'})
    await expect(response).to_be_ok()
    await page.reload()

    await grdm.open_edit_wiki(page, transition_timeout=transition_timeout)
    await expect(page.locator('#mEditor .ProseMirror[contenteditable="true"]')).to_have_text(
        initial_wiki_text, timeout=transition_timeout
    )

await run_pw(_step)

## 本文を変更せずに「閉じる」をクリックする

未保存の確認が表示されず、閲覧モードで「編集前の本文」が表示されること。

In [ ]:
wiki_editor_selector = '#mEditor .ProseMirror[contenteditable="true"]'

def wiki_editor(page):
    return page.locator(wiki_editor_selector)

async def expect_editor_text(page, text):
    await page.wait_for_function(
        """
        ({selector, expected}) => {
            const editor = document.querySelector(selector);
            if (!editor) return false;
            const body = editor.cloneNode(true);
            body.querySelectorAll('.ProseMirror-yjs-cursor').forEach(cursor => cursor.remove());
            return body.textContent === expected;
        }
        """,
        arg={'selector': wiki_editor_selector, 'expected': text},
        timeout=transition_timeout,
    )

async def open_editor(page, expected_text):
    await grdm.open_edit_wiki(page, transition_timeout=transition_timeout)
    await expect_editor_text(page, expected_text)

async def expect_wiki_view(page, text):
    await expect(page.locator('#editWysiwyg')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#wikiViewRender')).to_contain_text(text, timeout=transition_timeout)

async def _step(page):
    await page.locator('#revert-button').click()
    await expect_wiki_view(page, initial_wiki_text)
    await expect(page.locator('#closeConfirmModal')).to_be_hidden()

await run_pw(_step)

## 「編集」をクリックして本文を変更し、「閉じる」をクリックする

未保存の変更についての確認ダイアログが表示されること。

In [ ]:
discarded_text = '破棄する本文'

async def _step(page):
    await open_editor(page, initial_wiki_text)
    await wiki_editor(page).fill(discarded_text)
    await expect(wiki_editor(page)).to_have_text(discarded_text, timeout=transition_timeout)
    await page.locator('#revert-button').click()
    await expect(page.locator('#closeConfirmModal')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログの「キャンセル」をクリックする

確認ダイアログが閉じ、編集欄に変更した本文が残ること。

In [ ]:
async def _step(page):
    modal = page.locator('#closeConfirmModal')
    await modal.locator('button.btn-default[data-dismiss="modal"]').click()
    await expect(modal).to_be_hidden()
    await expect(wiki_editor(page)).to_have_text(discarded_text, timeout=transition_timeout)

await run_pw(_step)

## 「閉じる」をクリックし、確認ダイアログの「破棄」をクリックする

閲覧モードに戻り、保存済みの「編集前の本文」が表示されること。

In [ ]:
async def _step(page):
    modal = page.locator('#closeConfirmModal')
    await page.locator('#revert-button').click()
    await modal.locator('button.btn-danger').click()
    await expect(modal).to_be_hidden()
    await expect_wiki_view(page, initial_wiki_text)

await run_pw(_step)

## 「編集」をクリックして本文を変更し、「閉じる」から「保存して閉じる」を選ぶ

編集を始めると保存済みの本文が表示されること。「保存して閉じる」をクリックすると変更した本文が保存され、閲覧モードに戻ること。

In [ ]:
saved_text = '保存して閉じた本文'

async def _step(page):
    await open_editor(page, initial_wiki_text)
    await wiki_editor(page).fill(saved_text)
    await expect(wiki_editor(page)).to_have_text(saved_text, timeout=transition_timeout)
    await page.locator('#revert-button').click()
    modal = page.locator('#closeConfirmModal')
    async with page.expect_navigation(timeout=transition_timeout):
        await modal.locator('button.btn-success').click()
    await expect_wiki_view(page, saved_text)

await run_pw(_step)

## 「編集」をクリックして本文を変更し、フッターの「保存」をクリックする

保存して閉じた本文を再編集できること。保存時に離脱警告が出ず、変更内容が表示されること。

In [ ]:
footer_text = 'フッターから保存した本文'

async def _step(page):
    await open_editor(page, saved_text)
    await wiki_editor(page).fill(footer_text)
    await expect(wiki_editor(page)).to_have_text(footer_text, timeout=transition_timeout)
    dialogs = []

    async def record_dialog(dialog):
        dialogs.append(dialog.type)
        await dialog.accept()

    page.on('dialog', record_dialog)
    try:
        async with page.expect_navigation(timeout=transition_timeout):
            await page.locator('#mEditorFooter input[type="submit"][value="保存"]').click()
    finally:
        page.remove_listener('dialog', record_dialog)

    assert not dialogs, f'保存時に予期しないダイアログが表示されました: {dialogs}'
    await expect_wiki_view(page, footer_text)

await run_pw(_step)

## 「編集」をクリックして本文を変更し、ブラウザーでページの再読み込みを試みる

離脱警告が出ること。離脱を取りやめると編集欄に未保存の本文が残ること。

In [ ]:
unsaved_text = '離脱警告を確認する本文'

async def _step(page):
    await open_editor(page, footer_text)
    await wiki_editor(page).fill(unsaved_text)
    await expect(wiki_editor(page)).to_have_text(unsaved_text, timeout=transition_timeout)
    async def dismiss_warning(dialog):
        await dialog.dismiss()

    page.once('dialog', dismiss_warning)
    async with page.expect_event('dialog', timeout=transition_timeout) as warning:
        await page.evaluate('window.location.reload()')
    dialog = await warning.value
    assert dialog.type == 'beforeunload', dialog.type
    await expect(wiki_editor(page)).to_have_text(unsaved_text, timeout=transition_timeout)

await run_pw(_step)

## 「閉じる」をクリックし、確認ダイアログの「破棄」をクリックする

閲覧モードに戻り、保存済みの本文が表示されること。

In [ ]:
async def _step(page):
    await page.locator('#revert-button').click()
    await page.locator('#closeConfirmModal button.btn-danger').click()
    await expect_wiki_view(page, footer_text)

await run_pw(_step)

## （共同編集を実行する場合）同じWikiを2つのタブで編集する

タブAの変更がタブBの編集欄に反映され、タブBにタブAの編集者名が表示されること。

In [ ]:
peer = None
live_text_a = 'タブAの編集中の本文'

async def _step(page):
    global peer, cursor_a
    peer = await page.context.new_page()
    await peer.goto(page.url.split("?", 1)[0])
    await expect_wiki_view(peer, footer_text)
    await open_editor(page, footer_text)
    await open_editor(peer, footer_text)

    name = await page.evaluate('window.contextVars.currentUser.fullname')
    assert name
    cursor_a = peer.locator('#mEditor .ProseMirror-yjs-cursor div').filter(has_text=name)
    await page.bring_to_front()
    await wiki_editor(page).fill(live_text_a)
    await wiki_editor(page).press('End')
    await expect_editor_text(peer, live_text_a)
    await expect(cursor_a).to_be_visible(timeout=transition_timeout)

if not skip_wiki_collaboration:
    await run_pw(_step)
else:
    print('共同編集テストはスキップします (skip_wiki_collaboration=True)')

## （共同編集を実行する場合）タブAで「閉じる」をクリックし、タブBの本文を変更してからタブAを再編集する

タブAでは確認ダイアログが出ず、タブBでは編集を続けられること。タブAを再編集するとタブBの未保存の本文が見え、タブBにタブAの編集者名が再表示されること。

In [ ]:
live_text_b = 'タブBの編集中の本文'

async def _step(page):
    await page.locator('#revert-button').click()
    await expect(page.locator('#editWysiwyg')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#closeConfirmModal')).to_be_hidden()
    await expect(cursor_a).to_have_count(0, timeout=transition_timeout)

    await peer.bring_to_front()
    await wiki_editor(peer).fill(live_text_b)
    await expect_editor_text(peer, live_text_b)

    await page.bring_to_front()
    await open_editor(page, live_text_b)
    await wiki_editor(page).click()
    await wiki_editor(page).press('End')
    await expect(cursor_a).to_be_visible(timeout=transition_timeout)

if not skip_wiki_collaboration:
    await run_pw(_step)

## （共同編集を実行する場合）タブAで「閉じる」をクリックし、タブBで保存してからタブAを再編集する

タブBで保存した本文がタブAの編集欄にも表示されること。

In [ ]:
peer_saved_text = 'タブBが保存した本文'

async def _step(page):
    await page.locator('#revert-button').click()
    await expect(page.locator('#editWysiwyg')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#closeConfirmModal')).to_be_hidden()
    await expect(cursor_a).to_have_count(0, timeout=transition_timeout)

    await peer.bring_to_front()
    await wiki_editor(peer).fill(peer_saved_text)
    await expect_editor_text(peer, peer_saved_text)
    async with peer.expect_navigation(timeout=transition_timeout):
        await peer.locator('#mEditorFooter input[type="submit"][value="保存"]').click()
    await expect_wiki_view(peer, peer_saved_text)

    await page.bring_to_front()
    await open_editor(page, peer_saved_text)

if not skip_wiki_collaboration:
    await run_pw(_step)

## （共同編集を実行する場合）2つのタブで編集し、タブBを閉じてからタブAの変更を破棄し、再編集する

タブAで再編集したとき、タブBが保存した本文が表示され、破棄した変更が残っていないこと。

In [ ]:
async def _step(page):
    await peer.bring_to_front()
    await open_editor(peer, peer_saved_text)
    name = await peer.evaluate('window.contextVars.currentUser.fullname')
    assert name
    cursor_b = page.locator('#mEditor .ProseMirror-yjs-cursor div').filter(has_text=name)
    await wiki_editor(peer).click()
    await wiki_editor(peer).press('End')
    await expect(cursor_b).to_be_visible(timeout=transition_timeout)

    await page.bring_to_front()
    await wiki_editor(page).fill(discarded_text)
    await expect_editor_text(peer, discarded_text)

    await peer.bring_to_front()
    await peer.locator('#revert-button').click()
    await expect(peer.locator('#editWysiwyg')).to_be_visible(timeout=transition_timeout)
    await expect(peer.locator('#closeConfirmModal')).to_be_hidden()
    await expect(cursor_b).to_have_count(0, timeout=transition_timeout)
    await peer.close()

    await page.bring_to_front()
    await page.locator('#revert-button').click()
    await page.locator('#closeConfirmModal button.btn-danger').click()
    await expect_wiki_view(page, peer_saved_text)
    await open_editor(page, peer_saved_text)
    await page.locator('#revert-button').click()
    await expect_wiki_view(page, peer_saved_text)

if not skip_wiki_collaboration:
    await run_pw(_step)

## 新規Wikiを作成し、ページを再読み込みせずに本文を入力して保存する

入力した本文が編集欄に表示され、保存後の閲覧画面にも表示されること。

In [ ]:
async def _step(page):
    name = '新規Wikiの初回編集'
    body = 'first edit'
    await page.locator('div[data-target="#newWiki"]').click()
    await page.locator('input[name="addHierarchy"][value="same"]').check()
    await page.locator('#data').fill(name)
    await page.locator('#add-wiki-submit').click()
    await expect(page.locator('#pageName')).to_have_text(name, timeout=transition_timeout)

    await grdm.open_edit_wiki(page, transition_timeout=transition_timeout)
    editor = wiki_editor(page)
    await editor.click()
    await editor.press_sequentially(body)
    await expect(editor).to_have_text(body, timeout=transition_timeout)
    async with page.expect_navigation(timeout=transition_timeout):
        await page.locator('#mEditorFooter input[type="submit"][value="保存"]').click()
    await expect_wiki_view(page, body)

await run_pw(_step)

## (指定がある場合) プロジェクトを削除する

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await scripts.grdm.delete_project(page)
    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}